If you want to type along with me, head to [this notebook](https://humboldt.cloudbank.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fmlekimchi%2Fdata111_fall26&branch=main&urlpath=tree%2Fdata111_fall26%2Flectures%2Flec16_live.ipynb)
instead. If you prefer follow along by executing the cells, stay in this notebook.

# Lecture 16: Statistics and Assessing Models
v.ekc

Today we cross from probability into **inference**: a *parameter* describes a population, a *statistic* describes a sample — and simulating a statistic tells us what samples *should* look like, so we can judge models against reality. (Slides: KL Lecture 16 deck.)

**Today's sections:**
1. Random sampling, recapped
2. Distributions, recapped
3. Large random samples
4. Statistics and parameters
5. The empirical distribution of a statistic
6. Sample proportions and Swain v. Alabama

In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

---
## 1. Random Sampling, Recapped

The United flights again — deterministic samples first, then a random one. (This mirrors the end of Lecture 15; speed through if it's fresh.)

We load in a dataset of all United flights national flights from 6/1/15 to 8/9/15, their destination and how long they were delayed, in minutes.

In [ ]:
united = Table.read_table('united.csv')
united = united.with_column('Row', np.arange(united.num_rows)).move_to_start('Row')
united

Some deterministic samples:

In [ ]:
united.take(np.arange(10))

In [ ]:
united.where('Destination', 'JFK') 

In [ ]:
united.take(np.arange(0, united.num_rows, 1000))

In [ ]:
united.take(make_array(34, 6321, 10040))

A random sample:

In [ ]:
start = np.random.choice(np.arange(1000))
systematic_sample = united.take(np.arange(start, united.num_rows, 1000))
systematic_sample.show()

---
## 2. Distributions, Recapped

Probability distribution (the die's true 1/6-per-face) vs. empirical distribution (what your rolls actually did).

In [ ]:
die = Table().with_column('Face', np.arange(1, 7))
die

In [ ]:
die.sample(10)

In [ ]:
roll_bins = np.arange(0.5, 6.6, 1)

In [ ]:
die.hist(bins=roll_bins)

In [ ]:
die.sample(10).hist(bins=roll_bins)

### Try It 1: Law of Averages reps 😊

1. Simulate 1000 rolls and plot the empirical distribution. Compare to 10 rolls.

2. Now 10,000 rolls.

In [ ]:
# 1. 1000 rolls


In [ ]:
# 2. 10000 rolls


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*More rolls → flatter, closer to the true distribution.*

```python
# 1.
die.sample(1000).hist(bins=roll_bins)

# 2.
die.sample(10000).hist(bins=roll_bins)
```

</details>

---
## 3. Large Random Samples

Same story with real data — the population distribution of flight delays:

In [ ]:
united = Table.read_table('united.csv')
united = united.with_column('Row', np.arange(united.num_rows)).move_to_start('Row')
united

In [ ]:
united_bins = np.arange(-20, 201, 5)
united.hist('Delay', bins = united_bins)

In [ ]:
min(united.column('Delay'))

In [ ]:
max(united.column('Delay'))

In [ ]:
np.average(united.column('Delay'))

### Try It 2: Sample sizes 😊

1. Draw a sample of 10 flights and plot the delays. What do you observe?

2. Now 1000 flights.

In [ ]:
# 1. sample of 10


In [ ]:
# 2. sample of 1000


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Ten flights can look like anything; a thousand look like the population.*

```python
# 1.
united.sample(10).hist('Delay', bins=united_bins)

# 2.
united.sample(1000).hist('Delay', bins=united_bins)
```

</details>

---
## 4. Statistics and Parameters 🎯

The vocabulary of inference (KL deck, slides 11–16):

### 📋 Board Reference

| Term | Meaning | Example |
|---|---|---|
| **Parameter** | a number describing the **population** | median delay of *all* flights |
| **Statistic** | a number computed from a **sample** | median delay of 10 sampled flights |
| Key fact | statistics **vary** from sample to sample | run the cells below twice! |
| Inference | using a statistic to estimate a parameter | our project for the next 6 lectures |

In [ ]:
# (Population) Parameter
np.median(united.column('Delay'))

In [ ]:
# (Sample) Statistic
np.median(united.sample(10).column('Delay'))

In [ ]:
# (Sample) Statistic
np.median(united.sample(100).column('Delay'))

### Probability & Empirical Distributions of a Statistic

A statistic is random — so it *has a distribution*. Simulate the statistic many times and you can see it:

In [ ]:
def sample_median(size):
    return np.median(united.sample(size).column('Delay'))

In [ ]:
sample_median(10)

In [ ]:
num_simulations = 2000

In [ ]:
sample_medians = make_array()

for i in np.arange(num_simulations):
    new_median = sample_median(10)
    sample_medians = np.append(sample_medians, new_median)

In [ ]:
Table().with_column('Sample medians (size=10)', sample_medians).hist(bins=20)

In [ ]:
sample_medians = make_array()

for i in np.arange(num_simulations):
    new_median = sample_median(1000)
    sample_medians = np.append(sample_medians, new_median)

In [ ]:
Table().with_column('Sample medians (size=1K)', sample_medians).hist(bins = np.arange(-0.5,7.5))

#### Empirical Distributions of a Statistic (Overlayed)

In [ ]:
sample_medians_10 = make_array()
sample_medians_100 = make_array()
sample_medians_1000 = make_array()

num_simulations = 2000

for i in np.arange(num_simulations):
    new_median_10 = sample_median(10)
    sample_medians_10 = np.append(sample_medians_10, new_median_10)
    new_median_100 = sample_median(100)
    sample_medians_100 = np.append(sample_medians_100, new_median_100)
    new_median_1000 = sample_median(1000)
    sample_medians_1000 = np.append(sample_medians_1000, new_median_1000)

In [ ]:
sample_medians = Table().with_columns('Size 10', sample_medians_10, 
                                      'Size 100', sample_medians_100,
                                      'Size 1000', sample_medians_1000)

In [ ]:
sample_medians.hist(bins = np.arange(-5, 30))

> **Bigger samples → tighter statistic.** The size-1000 medians barely budge; the size-10 medians are all over the place. Sample size buys reliability.

---
## 5. Sample Proportions

`sample_proportions(n, dist)` simulates drawing `n` individuals from a categorical distribution and returns the sampled *proportions* — one function that replaces a whole `for` loop. You'll use it constantly (Lab 6, Homework 6).

## Sample Proportions

Say I have an unfair coin. With 40% chance of flipping heads and 60% chance of flipping tails.

In [ ]:
unfair_coin_proportions = make_array(0.4,0.6)
sample_proportions(100,unfair_coin_proportions)

---
## 6. Swain v. Alabama ⚖️

In Talladega County, Alabama, 1965, 26% of eligible jurors were Black men — yet Robert Swain's 100-person jury panel had only 8. The Supreme Court called this acceptable. Let's ask the data: **if panels really were drawn from the eligible population, how many Black men would a panel of 100 have?** (KL deck, slides 22–27.)

In [ ]:
population_proportions = make_array(.26, .74)
population_proportions

In [ ]:
sample_proportions(100, population_proportions)

In [ ]:
def panel_proportion():
    return sample_proportions(100, population_proportions).item(0)

In [ ]:
panel_proportion()

In [ ]:
panels = make_array()

for i in np.arange(10000):
    new_panel = panel_proportion() * 100
    panels = np.append(panels, new_panel)

In [ ]:
Table().with_column(
    'Number of Black Men on Panel of 100', panels
).hist(bins=np.arange(5.5,40.))

# Plotting details; ignore this code
plots.ylim(-0.002, 0.09)
plots.scatter(8, 0, color='red', s=30);

> **Read the histogram:** simulated panels essentially *never* dip to 8 — the observed value is far outside anything chance produces. The model "panels are random draws from the eligible population" is not believable. That's assessing a model with a statistic — the blueprint for the next three lectures.

---
> **Today's takeaway:** parameters belong to populations, statistics to samples; statistics have distributions you can simulate; and when the observed statistic sits far outside the simulated distribution, the model is in trouble.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### Minimal template — simulate a statistic
```python
def one_statistic():
    return np.median(population.sample(n).column('col'))

stats = make_array()
for i in np.arange(10000):
    stats = np.append(stats, one_statistic())
Table().with_column('Statistic', stats).hist()
```